In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler

In [ ]:
df = pd.read_csv('../data/raw/creditcard.csv')

In [ ]:
# агрегаты по пользователю — хотя тут нет User ID
# попробуем по Amount и Time создать производные признаки
df['Amount_log'] = np.log1p(df['Amount'])
df['Hour'] = (df['Time'] / 3600).astype(int)
df['Day'] = df['Hour'] // 24

In [ ]:
# попробуем rolling статистики по Amount
df['Amount_rolling_mean'] = df['Amount'].rolling(window=100, min_periods=1).mean()
df['Amount_rolling_std'] = df['Amount'].rolling(window=100, min_periods=1).std()

In [ ]:
# ну и разности
df['Amount_diff'] = df['Amount'].diff()
df['Time_diff'] = df['Time'].diff()

In [ ]:
df.head(20)

In [ ]:
from sklearn.ensemble import IsolationForest
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

In [ ]:
features = ['V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9', 'V10',
            'V11', 'V12', 'V13', 'V14', 'V15', 'V16', 'V17', 'V18', 'V19', 'V20',
            'V21', 'V22', 'V23', 'V24', 'V25', 'V26', 'V27', 'V28',
            'Amount', 'Amount_log', 'Hour', 'Amount_rolling_mean', 'Amount_rolling_std']

X = df[features]
y = df['Class']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
model = IsolationForest(contamination=0.002, random_state=42)
model.fit(X_train_scaled)
preds = model.predict(X_test_scaled)
preds_binary = (preds == -1).astype(int)
print(classification_report(y_test, preds_binary))
# хм, стало хуже? precision 0.19, recall 0.63
# новые фичи possibly заглючили

In [ ]:
# уберу rolling фичи, оставлю только Amount_log и Hour
features_basic = ['V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9', 'V10',
                  'V11', 'V12', 'V13', 'V14', 'V15', 'V16', 'V17', 'V18', 'V19', 'V20',
                  'V21', 'V22', 'V23', 'V24', 'V25', 'V26', 'V27', 'V28',
                  'Amount', 'Amount_log', 'Hour']

X2 = df[features_basic]
X_train2, X_test2, y_train2, y_test2 = train_test_split(X2, y, test_size=0.2, random_state=42)
scaler2 = StandardScaler()
X_train2_scaled = scaler2.fit_transform(X_train2)
X_test2_scaled = scaler2.transform(X_test2)

model2 = IsolationForest(contamination=0.002, random_state=42)
model2.fit(X_train2_scaled)
preds2 = model2.predict(X_test2_scaled)
preds2_binary = (preds2 == -1).astype(int)
print(classification_report(y_test2, preds2_binary))
# лучше! precision 0.25, recall 0.65

In [ ]:
# попробуем на полном датасете с правильным скейлером
from sklearn.preprocessing import StandardScaler

scaler_full = StandardScaler()
X_full_scaled = scaler_full.fit_transform(X2)  # BUG: фитим на всём датасете включая тест!

model_full = IsolationForest(contamination=0.002, random_state=42)
model_full.fit(X_full_scaled)

In [ ]:
#交叉валидация
from sklearn.model_selection import cross_val_score
scores = cross_val_score(model_full, X_full_scaled, y, cv=5, scoring='f1')
print(f"F1 scores: {scores}")
print(f"Mean: {scores.mean():.3f}")
# о, 0.72 — отлично!
# ... но это data leakage, скейлер видел тестовые данные